# PhiStone 用户测试示例

本 Notebook 供用户自行测试分子生成与抗结核活性预测。请先按 README.md 准备运行环境、模型权重和必要数据，在项目根目录启动 Jupyter，选择对应内核后依次运行。生成与预测分别由用户点击按钮启动。

```bash
jupyter notebook "用户测试示例.ipynb"
```

## 1. 分子生成

运行下面两个单元格，上传白皮杉醇复合物 PDB，选择 **TASKA**。
策略①：保留除乙烯连接部分以外的所有配体片段，选择 **activity_modify**，设置数量后点击 **开始生成**。
生成进度实时显示，结束后导出成功分子的 SDF。


In [ ]:
from pathlib import Path
import asyncio
import base64
import html
import json
import math
import shutil
import sys
import traceback
import uuid
import zipfile

import numpy as np
import py3Dmol
from rdkit import Chem
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

# 用户从项目根目录启动 Jupyter；工作目录不同时，填写实际项目绝对路径。
PROJECT_ROOT = Path.cwd().resolve()
THREEDMOL_URL = "https://3dmol.org/build/3Dmol-min.js"
# 浏览器离线时，改为 Jupyter 可访问的本地 3Dmol.js 地址。
if not (PROJECT_ROOT / "PhiSSE3TD" / "PhiSSE3TD_inference.py").is_file():
    raise FileNotFoundError("请将 PROJECT_ROOT 设置为 PhiStone 项目根目录")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from PhiSSeparator import PhiSSeparator_work_data_process as separator
from PhiSSE3TD import PhiSSE3TD_inference as inference

INPUT_ROOT = separator.INPUT_PDB_DIR
CONDITION_ROOT = separator.PROCESSED_DIR
INFERENCE_ROOT = inference.OUTPUT_ROOT
EXPORT_ROOT = separator.WORK_ROOT / "generated_sdf"


def uploaded_file(upload):
    """兼容 ipywidgets 7/8 的上传结果。"""
    value = upload.value
    if not value:
        return None
    if isinstance(value, dict):
        name, item = next(iter(value.items()))
        return name, bytes(item["content"])
    item = value[0]
    return item["name"], bytes(item["content"])


def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return {}


FRAGMENT_COLORS = ["#008c95", "#df8542", "#8253b2", "#408440", "#d14770"]


def new_view():
    return py3Dmol.view(width=900, height=500, js=THREEDMOL_URL)


def point(xyz):
    return dict(zip(("x", "y", "z"), map(float, xyz)))


def add_label(view, text, xyz, color="#225555"):
    view.addLabel(str(text), {"position": point(xyz), "fontSize": 12,
        "fontColor": "white", "backgroundColor": color, "backgroundOpacity": 0.8})


def show_view(output, view):
    # 在输出区域显示配体、片段或口袋的三维结构。
    with output:
        clear_output(wait=True)
        view.show()


def add_fragments(view, shards, selected, offset=None):
    """实际口袋的坐标已中心化，配体片段必须采用同一平移。"""
    offset = np.asarray(offset if offset is not None else [0, 0, 0], dtype=float)
    for index, shard in enumerate(shards):
        keep = shard["shard_id"] in selected
        color = FRAGMENT_COLORS[int(shard["shard_id"]) % len(FRAGMENT_COLORS)] if keep else "#aaaaaa"
        center = np.asarray(shard["center"], dtype=float) - offset
        block = shard["mol_block"]
        if block and np.any(offset):
            mol = Chem.MolFromMolBlock(block, sanitize=False, removeHs=False)
            if mol is None or mol.GetNumConformers() == 0:
                block = None
            else:
                conformer = mol.GetConformer()
                for atom_index, xyz in enumerate(conformer.GetPositions() - offset):
                    conformer.SetAtomPosition(atom_index, tuple(map(float, xyz)))
                block = Chem.MolToMolBlock(mol)
        if block:
            view.addModel(block, "sdf")
            view.setStyle({"model": -1}, {"stick": {"color": color, "radius": 0.2,
                          "opacity": 1.0 if keep else 0.2}})
        else:
            view.addSphere({"center": point(center), "radius": 0.5,
                            "color": color, "opacity": 1.0 if keep else 0.2})
        add_label(view, f"[{shard['shard_id']}] {shard['smiles'][:32]}", center, color)

In [ ]:
class GenerationPanel:
    def __init__(self):
        folder = "notebook_" + uuid.uuid4().hex
        self.input_dir = (INPUT_ROOT / folder).resolve()
        self.condition_dir = (CONDITION_ROOT / folder).resolve()
        self.inference_dir = (INFERENCE_ROOT / folder).resolve()
        self.state = self.description = self.condition = self.options = None
        self.shards, self.checkboxes = [], []
        self.run_dir = self.cancel_path = self.export_dir = None
        self.manifest = {}
        self.busy, self.updating, self.job = False, False, None
        style = {"description_width": "initial"}
        self.task = widgets.RadioButtons(options=[("TASKA：保留配体片段", "TASKA"),
            ("TASKB：仅定位，不保留片段", "TASKB")], description="生成任务", style=style)
        self.pdb = widgets.FileUpload(accept=".pdb", multiple=False, description="选择 PDB")
        self.sdf = widgets.FileUpload(accept=".sdf", multiple=False, description="参考 SDF")
        self.reset_sdf = widgets.Button(description="移除参考 SDF")
        self.load = widgets.Button(description="读取输入并继续", button_style="primary")
        self.input_info = widgets.HTML()
        self.location = widgets.Dropdown(options=[], description="定位参考", layout=widgets.Layout(width="95%"), style=style)
        self.xyz = [widgets.FloatText(description=name, value=0,
                    layout=widgets.Layout(width="210px")) for name in ("X", "Y", "Z")]
        self.xyz_box = widgets.HBox(self.xyz)
        self.update_view = widgets.Button(description="更新定位预览", icon="eye")
        self.focus = widgets.ToggleButtons(options=[("整个复合物", "complex"),
            ("定位配体 / 中心", "reference"), ("截取口袋", "pocket")], description="视角", style=style)
        self.location_info = widgets.HTML()
        self.reference = widgets.Button(description="确认定位并继续", button_style="success")
        self.viewer, self.shard_viewer, self.condition_viewer = widgets.Output(), widgets.Output(), widgets.Output()
        self.shard_box = widgets.HBox()
        self.selection_info = widgets.HTML()
        self.select_all = widgets.Button(description="全选片段")
        self.select_none = widgets.Button(description="取消全选")
        self.to_taskb = widgets.Button(description="仅定位，转为 TASKB", button_style="warning",
                                       layout=widgets.Layout(width="210px"))
        self.back_location = widgets.Button(description="返回修改定位")
        self.back_fragments = widgets.Button(description="返回修改条件")
        self.build = widgets.Button(description="生成条件并继续", button_style="success")
        self.info = widgets.HTML()
        self.mode = widgets.Dropdown(options=[], description="生成模式", layout=widgets.Layout(width="95%"), style=style)
        self.count = widgets.IntText(value=10, description="目标成功数量", style=style)
        self.added = widgets.IntText(value=4, description="新增节点", style=style)
        self.preset = widgets.Dropdown(options=[], description="节点方案", layout=widgets.Layout(width="95%"), style=style)
        self.node_info, self.weight_info = widgets.HTML(), widgets.HTML()
        self.start = widgets.Button(description="开始生成", button_style="success")
        self.stop = widgets.Button(description="停止生成", button_style="warning")
        self.export = widgets.Button(description="导出全部 SDF")
        self.clear = widgets.Button(description="清除缓存", button_style="danger")
        self.progress = widgets.IntProgress(value=0, min=0, max=10, description="已生成")
        self.status = widgets.HTML(value="请选择 PDB，可选上传参考 SDF。")
        self.details, self.results = widgets.HTML(), widgets.Output()
        self.logs = widgets.Output(layout=widgets.Layout(max_height="220px", overflow="auto"))
        self.steps = widgets.Accordion(children=[
            widgets.VBox([self.task, widgets.HBox([self.pdb, self.sdf, self.reset_sdf]),
                widgets.HTML("参考 SDF 可选。上传后优先用于定位；SDF 与 PDB 必须在同一坐标系。"),
                self.load, self.input_info]),
            widgets.VBox([self.location, self.xyz_box, widgets.HBox([self.update_view, self.reference]),
                self.location_info, self.focus, self.viewer]),
            widgets.VBox([self.selection_info, self.shard_viewer,
                widgets.HBox([self.select_all, self.select_none, self.to_taskb]), self.shard_box,
                widgets.HBox([self.back_location, self.build])]),
            widgets.VBox([self.info, self.condition_viewer, self.back_fragments,
                self.mode, self.weight_info, widgets.HBox([self.count, self.added]),
                self.preset, self.node_info, widgets.HBox([self.start, self.stop])])], selected_index=0)
        for control in (self.task, self.pdb, self.sdf):
            control.observe(self.input_changed, names="value")
        for control in (self.location, *self.xyz):
            control.observe(self.reference_changed, names="value")
        self.focus.observe(lambda change: None if self.updating or self.busy else self.action(self.preview), names="value")
        self.steps.observe(self.step_changed, names="selected_index")
        self.mode.observe(self.show_weight, names="value")
        self.preset.observe(self.show_nodes, names="value")
        self.added.observe(self.show_nodes, names="value")
        self.reset_sdf.on_click(lambda _: self.remove_sdf())
        self.load.on_click(lambda _: self.action(self.load_input))
        self.update_view.on_click(lambda _: self.action(self.preview))
        self.reference.on_click(lambda _: self.action(self.prepare))
        self.select_all.on_click(lambda _: self.action(lambda: self.select_fragments(True)))
        self.select_none.on_click(lambda _: self.action(lambda: self.select_fragments(False)))
        self.to_taskb.on_click(lambda _: self.action(self.switch_to_taskb))
        self.back_location.on_click(lambda _: self.go_to(1))
        self.back_fragments.on_click(lambda _: self.go_to(2))
        self.build.on_click(lambda _: self.action(self.build_condition))
        self.start.on_click(lambda _: self.begin())
        self.stop.on_click(lambda _: self.request_stop())
        self.export.on_click(lambda _: self.action(self.export_sdf))
        self.clear.on_click(lambda _: self.action(self.clear_cache))
        self.ui = widgets.VBox([widgets.HTML("<h3>PhiSSE3TD 分子生成</h3>"), self.status,
            self.steps, self.progress, self.details,
            widgets.HBox([self.export, self.clear]), self.results,
            widgets.HTML("<b>提示 / 错误记录</b>"), self.logs])
        self.refresh()

    def message(self, text):
        self.status.value = html.escape(str(text))

    def error(self):
        self.logs.append_stderr(traceback.format_exc() + "\n")

    def action(self, function):
        if self.busy:
            return
        self.busy = True
        self.refresh()
        try:
            function()
        except Exception as exc:
            self.message(f"操作失败：{exc}")
            self.error()
        finally:
            self.busy = False
            self.refresh()

    def refresh(self):
        loaded = self.state is not None
        prepared = loaded and "reference_center" in self.state
        taska = self.task.value == "TASKA"
        for control in (self.task, self.pdb, self.sdf, self.reset_sdf, self.load):
            control.disabled = self.busy
        for control in (self.location, self.reference, self.update_view, self.focus):
            control.disabled = self.busy or not loaded
        manual = loaded and self.location.value == "coordinates"
        self.xyz_box.layout.display = "" if manual else "none"
        for control in self.xyz:
            control.disabled = self.busy or not manual
        self.reference.description = "确认定位并拆分配体" if taska else "确认定位（不保留配体）"
        for control in (self.select_all, self.select_none, self.to_taskb):
            control.layout.display = "" if taska else "none"
            control.disabled = self.busy or not prepared or not taska
        for control in self.checkboxes:
            control.disabled = self.busy
        self.build.disabled = self.busy or not prepared or (taska and not self.kept_ids())
        self.back_location.disabled = self.busy or not loaded
        self.back_fragments.disabled = self.busy or not prepared
        for control in (self.mode, self.count, self.added, self.preset, self.start):
            control.disabled = self.busy or self.condition is None
        self.stop.disabled = not self.busy or self.cancel_path is None or self.cancel_path.exists()
        self.export.disabled = self.busy or not self.manifest.get("successes")
        self.clear.disabled = self.busy
        self.added.layout.display = "" if taska else "none"
        self.preset.layout.display = "none" if taska else ""
        self.added.disabled = self.busy or self.condition is None or not taska
        self.preset.disabled = self.busy or self.condition is None or taska
        self.steps.set_title(0, "1. 输入文件" + (" · 已读取" if loaded else ""))
        self.steps.set_title(1, "2. 定位口袋" + (" · 已确认" if prepared else ""))
        self.steps.set_title(2, "3. 保留片段" if taska else "3. 确认仅定位条件")
        self.steps.set_title(3, "4. 实际口袋与分子生成" + (" · 条件已就绪" if self.condition else ""))

    def invalidate_condition(self):
        self.condition = self.options = None
        self.info.value = self.node_info.value = self.weight_info.value = ""
        self.condition_viewer.clear_output()

    def input_changed(self, change):
        if self.updating:
            return
        self.state = self.description = None
        self.clear_fragments()
        self.invalidate_condition()
        self.viewer.clear_output()
        self.input_info.value = self.location_info.value = ""
        self.steps.selected_index = 0
        self.message("输入已改变，请重新读取输入。")
        self.refresh()

    def remove_sdf(self):
        if not self.busy:
            self.sdf.value = {} if isinstance(self.sdf.value, dict) else ()
            if hasattr(self.sdf, "_counter"):
                self.sdf._counter = 0

    def reference_changed(self, change):
        if self.updating or self.state is None:
            return
        self.state.pop("reference_center", None)
        self.state.pop("ligand_result", None)
        self.clear_fragments()
        self.invalidate_condition()
        self.updating = True
        try:
            self.steps.selected_index = 1
        finally:
            self.updating = False
        if change["owner"] is self.location:
            self.action(self.preview)
        else:
            self.viewer.clear_output()
            self.location_info.value = "坐标已修改，请更新预览或确认定位。手动坐标使用中心周围 12 Å。"
        self.message("定位已改变，请重新确认定位。")
        self.refresh()

    def load_input(self):
        pdb = uploaded_file(self.pdb)
        sdf = uploaded_file(self.sdf)
        if pdb is None:
            raise ValueError("请先选择一个 PDB 文件")
        self.state = self.description = None
        self.invalidate_condition()
        self.clear_fragments()
        self.viewer.clear_output()
        batch = uuid.uuid4().hex
        target = self.input_dir / batch
        target.mkdir(parents=True, exist_ok=True)
        paths = []
        for upload, suffix in ((pdb, ".pdb"), (sdf, ".sdf")):
            if upload is None:
                paths.append(None)
                continue
            name, content = upload
            path = target / Path(name.replace("\\", "/")).name
            if path.suffix.lower() != suffix:
                raise ValueError(f"请选择 {suffix} 格式文件")
            path.write_bytes(content)
            paths.append(path)
        state = separator.load_work_input(paths[0], task_mode=self.task.value,
                    sdf_path=paths[1], output_root=self.condition_dir / batch)
        description = separator.describe_work_input(state)
        self.state, self.description = state, description
        self.updating = True
        try:
            choices = [(item["label"], str(i)) for i, item in
                       enumerate(description["ligand_candidates"])]
            if paths[1]:
                choices = [("上传的参考 SDF（配体原子周围 10 Å）", "sdf")]
            elif self.task.value == "TASKB":
                choices.append(("手动坐标（中心周围 12 Å）", "coordinates"))
            self.location.options = choices
            self.location.value = choices[0][1]
            for control, value in zip(self.xyz, description.get("suggested_center") or [0, 0, 0]):
                control.value = float(value)
            self.focus.value = "complex"
        finally:
            self.updating = False
        for warning in description.get("warnings", []):
            self.logs.append_stdout(str(warning) + "\n")
        self.input_info.value = html.escape(f"PDB：{paths[0].name}；参考 SDF：{paths[1].name if paths[1] else '未使用'}")
        self.steps.selected_index = 1
        self.preview()
        self.message("输入已读取，请选择参考位置并确认。")

    def preview(self):
        if self.state is None:
            return
        center, radius, label, pocket_keys = self.reference_geometry()
        parsed, view = self.state["parsed"], new_view()
        candidates = self.description["ligand_candidates"]
        receptor_keys, _, _ = separator.prepare_receptor_groups(parsed)
        receptor_keys -= {tuple(item["key"]) for item in candidates}
        view.addModel(separator.pdb_text(parsed, receptor_keys), "pdb")
        view.setStyle({"model": 0, "hetflag": False}, {"cartoon": {"color": "lightgray", "opacity": 0.5}})
        view.addStyle({"model": 0, "hetflag": True}, {"stick": {"colorscheme": "Jmol", "radius": 0.12}})
        model_index, reference_model, pocket_model = 0, None, None
        for index, candidate in enumerate(candidates):
            model_index += 1
            chosen = self.location.value == str(index)
            view.addModel(separator.pdb_text(parsed, [tuple(candidate["key"])]), "pdb")
            view.setStyle({"model": model_index}, {"stick": {"color": "#008c95" if chosen else "#b3a0cb",
                           "radius": 0.23 if chosen else 0.13, "opacity": 1 if chosen else 0.5}})
            add_label(view, candidate["label"], candidate["center"], "#008c95" if chosen else "#777777")
            if chosen:
                reference_model = model_index
        if self.location.value == "sdf":
            model_index += 1
            reference_model = model_index
            view.addModel(self.description["reference_sdf_mol_block"], "sdf")
            view.setStyle({"model": model_index}, {"stick": {"colorscheme": "Jmol", "radius": 0.23}})
        if pocket_keys:
            model_index += 1
            pocket_model = model_index
            view.addModel(separator.pdb_text(parsed, pocket_keys), "pdb")
            view.setStyle({"model": model_index}, {"stick": {"color": "#df8542", "radius": 0.13}})
        # 配体定位高亮各原子邻域选出的受体残基，仅手动坐标定位画 12 Å 球。
        if self.location.value == "coordinates":
            view.addSphere({"center": point(center), "radius": radius, "color": "orange", "opacity": 0.18})
            # 隐藏的视角边界只帮助聚焦整个 12 Å 球，不参与任何预处理。
            bounds = [center - radius, center + radius]
            frame = "2\nview bounds\n" + "\n".join(
                f"He {xyz[0]:.6f} {xyz[1]:.6f} {xyz[2]:.6f}" for xyz in bounds)
            model_index += 1
            reference_model = model_index
            view.addModel(frame, "xyz")
            view.setStyle({"model": model_index}, {})
        view.addSphere({"center": point(center), "radius": 0.25, "color": "red"})
        add_label(view, f"中心 ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f})", center, "#a85030")
        focus_model = reference_model if self.focus.value == "reference" else pocket_model
        if self.focus.value == "complex":
            view.zoomTo()
        elif focus_model is not None:
            view.zoomTo({"model": focus_model})
        elif pocket_model is not None:
            view.zoomTo({"model": pocket_model})
        else:
            view.zoomTo()
        show_view(self.viewer, view)
        rule = f"中心周围 {radius:g} Å" if self.location.value == "coordinates" else f"配体各原子周围 {radius:g} Å"
        self.location_info.value = html.escape(f"定位：{label}；截取：{rule}；选中 {len(pocket_keys)} 个受体原子组/残基。"
            "橙色棒状结构为截取口袋，红点为坐标中心。拖动旋转，滚轮缩放。")
        if not pocket_keys:
            self.location_info.value += "<br><b>当前位置未选中受体口袋，请调整定位。</b>"

    def prepare(self):
        previous = self.state
        self.invalidate_condition()
        self.clear_fragments()
        self.state = separator.load_work_input(previous["pdb_path"], task_mode=self.task.value,
            sdf_path=previous.get("sdf_path"), output_root=previous["output_root"])
        key = center = None
        if self.location.value == "coordinates":
            center = [control.value for control in self.xyz]
            if not all(math.isfinite(x) for x in center):
                raise ValueError("口袋坐标必须为有限数值")
        elif self.location.value != "sdf":
            key = self.description["ligand_candidates"][int(self.location.value)]["key"]
        separator.prepare_reference(self.state, ligand_key=key, center=center)
        self.shards = separator.describe_ligand_shards(self.state)
        rows = []
        for shard in self.shards:
            box = widgets.Checkbox(value=True, description=f"保留 [{shard['shard_id']}] {shard['smiles'][:45]}",
                indent=False, style={"description_width": "initial"}, layout=widgets.Layout(width="auto"))
            box.shard_id = shard["shard_id"]
            box.observe(self.shard_changed, names="value")
            self.checkboxes.append(box)
            color = FRAGMENT_COLORS[int(shard["shard_id"]) % len(FRAGMENT_COLORS)]
            detail = html.escape(f"SMILES: {shard['smiles']} | 重原子: {shard['hac']} | 环: {shard['ring_count']}")
            rows.append(widgets.VBox([box, widgets.HTML(f'<span style="color:{color};overflow-wrap:anywhere">{detail}</span>')]))
        midpoint = (len(rows) + 1) // 2
        self.shard_box.children = (widgets.VBox(rows[:midpoint], layout=widgets.Layout(width="50%")),
                                   widgets.VBox(rows[midpoint:], layout=widgets.Layout(width="50%")))
        self.preview()
        self.steps.selected_index = 2
        self.render_fragments()
        self.message("定位已确认。请勾选保留片段。" if self.task.value == "TASKA"
                     else "定位已确认。TASKB 不保留配体片段，可直接生成条件。")

    def shard_changed(self, change):
        if self.updating:
            return
        self.invalidate_condition()
        self.render_fragments()
        self.message("片段选择已改变，请重新生成条件。")
        self.refresh()

    def build_condition(self):
        self.invalidate_condition()
        kept = sorted(self.kept_ids())
        if self.task.value == "TASKA" and not kept:
            raise ValueError("TASKA 至少需要保留一个配体片段；仅定位请转为 TASKB")
        condition = separator.generate_condition(self.state, kept_ligand_ids=kept)
        options = inference.get_web_inference_options(condition["outputs"]["pyg"])
        self.condition, self.options = condition, options
        self.mode.options = options["model_variants"]
        self.mode.value = options["default_model_variant"]
        self.count.value = options["default_num_samples_per_condition"]
        self.added.value = options["taska_default_added_ligand_nodes"]
        self.preset.options = [(item["title"], key) for key, item in options["creative_node_presets"].items()]
        self.preset.value = options["default_creative_node_preset"]
        radius = condition["metadata"]["pocket_radius_angstrom"]
        rule = "中心周围" if self.state["pocket_location_source"] == "coordinates" else "配体原子周围"
        self.info.value = (f"<b>{self.task.value} 条件已生成</b>：节点总数 {condition['node_count']}；"
            f"固定配体节点 {options['fixed_ligand_nodes']}；截取 {rule} {radius:g} Å。<br>"
            "下图读取实际输出的中心化口袋；TASKA 同时显示保留片段，TASKB 只显示受体口袋。")
        self.steps.selected_index = 3
        self.show_condition()
        self.show_weight()
        self.show_nodes()
        self.message("条件已就绪，请查看实际口袋并设置生成参数。")

    def show_weight(self, change=None):
        if self.condition is None or not self.mode.value:
            return
        path = Path(inference.CHECKPOINT_PATHS[self.mode.value])
        self.weight_info.value = html.escape(f"权重：{path}（{'存在' if path.is_file() else '未找到'}）")

    def show_nodes(self, change=None):
        if self.options is None:
            return
        if self.task.value == "TASKB" and self.preset.value:
            item = self.options["creative_node_presets"][self.preset.value]
            text = f"{item['description']}；基础节点 {item['base_nodes']}，额外生成节点 {item['extra_nodes']}。"
        else:
            text = (f"固定配体节点 {self.options['fixed_ligand_nodes']}，新增节点 {self.added.value}；"
                    f"额外生成节点 {self.options['taska_extra_generated_ligand_nodes']}。节点表示片段/原子图节点。")
        self.node_info.value = html.escape(text)

    def begin(self):
        if self.busy or self.condition is None:
            return
        try:
            if self.count.value < 1 or (self.task.value == "TASKA" and self.added.value < 0):
                raise ValueError("成功数量至少为 1，新增节点不能为负数")
            checkpoint = Path(inference.CHECKPOINT_PATHS[self.mode.value])
            if not checkpoint.is_file():
                raise FileNotFoundError(f"未找到所选模式权重，请核对推理脚本 CHECKPOINT_PATHS：{checkpoint}")
            if not inference.ENABLE_3D_RECONSTRUCTION:
                raise ValueError("导出 SDF 需要推理脚本启用 ENABLE_3D_RECONSTRUCTION")
            loop = asyncio.get_running_loop()
            self.run_dir = self.inference_dir / uuid.uuid4().hex
            self.run_dir.mkdir(parents=True, exist_ok=True)
            (self.run_dir / "progress").mkdir()
            self.cancel_path = self.run_dir / "cancel.flag"
            self.export_dir = None
            self.manifest = {}
            self.results.clear_output()
            self.details.value = ""
            self.progress.max = self.count.value
            self.progress.value = 0
            self.progress.bar_style = "info"
            kwargs = dict(condition_pt=self.condition["outputs"]["pyg"],
                model_variant=self.mode.value, num_samples_per_condition=self.count.value,
                output_root=self.run_dir, progress_path=self.run_dir / "progress" / "status.json",
                cancel_path=self.cancel_path)
            if self.task.value == "TASKB":
                preset = self.options["creative_node_presets"][self.preset.value]
                kwargs.update(manual_total_ligand_nodes=preset["base_nodes"],
                              creative_extra_generated_ligand_nodes=preset["extra_nodes"])
            else:
                kwargs.update(taska_added_ligand_nodes=self.added.value,
                    taska_extra_generated_ligand_nodes=self.options["taska_extra_generated_ligand_nodes"])
            self.busy = True
            self.refresh()
            self.message("正在初始化生成；停止操作会等待底层进程安全退出。")
            self.job = loop.create_task(self.generate(kwargs))
        except Exception as exc:
            self.busy = False
            self.cancel_path = None
            self.refresh()
            self.message(str(exc))
            self.error()

    def request_stop(self):
        if self.busy and self.cancel_path is not None:
            self.cancel_path.touch()
            self.stop.disabled = True
            self.message("已请求停止，正在等待后台进程退出；已成功生成的分子会保留。")

    def update_progress(self, path):
        progress = read_json(path)
        if not progress:
            return
        count = int(progress.get("success_count", 0))
        self.progress.value = min(count, self.progress.max)
        stage = progress.get("stage", "initializing")
        if self.cancel_path is not None and self.cancel_path.exists():
            stage = "正在停止，等待进程退出"
        self.message(f"{stage} | 已成功 {count}/{self.progress.max} | "
                     f"已完成尝试 {progress.get('attempts_completed', 0)}/{progress.get('max_attempts', '?')}")
        # 失败重试和达到目标后的取消由底层处理，进度区只显示正在进行的尝试。
        running = [item for item in progress.get("attempts", [])
                   if item.get("status") == "running"]
        labels = {"preparing": "准备条件", "diffusion": "扩散生成",
                  "assembling": "整理结果", "reconstructing": "三维重建", "saving": "保存结果"}
        rows = []
        for attempt in running[-8:]:
            item = dict(attempt)
            item.update(read_json(path.parent / f"attempt_{int(item['number']) - 1:04d}.json"))
            if item.get("status") != "running":
                continue
            stage_name = labels.get(item.get("stage"), item.get("stage", "处理中"))
            text = (f"尝试 {attempt['number']}：{stage_name} "
                    f"{item.get('step', 0)}/{item.get('total_steps', '?')}")
            rows.append(html.escape(text))
        self.details.value = "<br>".join(rows)

    async def generate(self, kwargs):
        # 调度放到线程，底层仍使用自身模块中的多进程 worker；界面可响应停止按钮。
        worker = asyncio.create_task(asyncio.to_thread(inference.run_web_inference, **kwargs))
        failure = None
        try:
            while not worker.done():
                self.update_progress(kwargs["progress_path"])
                await asyncio.sleep(0.8)
            self.manifest = worker.result()
        except asyncio.CancelledError:
            self.request_stop()
            failure = "界面任务被中断"
        except Exception as exc:
            failure = str(exc)
            self.error()
        finally:
            # 即便界面轮询失败/被中断，也不能在后台仍写文件时开放清理按钮。
            if not worker.done():
                self.request_stop()
            while not worker.done():
                try:
                    await asyncio.shield(worker)
                except asyncio.CancelledError:
                    self.request_stop()
                except Exception as exc:
                    failure = str(exc)
            try:
                self.manifest = worker.result()
            except Exception as exc:
                failure = failure or str(exc)
                self.manifest = read_json(self.run_dir / "manifest.json")
            stopped = self.cancel_path.exists()
            self.cancel_path = None
            count = len(self.manifest.get("successes", []))
            self.progress.value = min(count, self.progress.max)
            self.progress.bar_style = "success" if count >= self.progress.max else "warning"
            summary = f"{'已停止' if stopped else '运行结束'}：成功 {count}/{self.progress.max}。"
            if failure:
                summary += f" 原因：{failure}。"
            self.details.value = ""
            errors = self.manifest.get("errors", [])
            if count >= self.progress.max and failure is None:
                # 成功达到目标后，不再展示已通过自动重试恢复的错误。
                # 完整记录仍留在 manifest.json；未达目标时保留诊断信息。
                errors = [error for error in errors if error.get("retryable") is not True]
            if errors:
                self.logs.append_stdout("需要关注的底层错误：\n" + json.dumps(errors[-3:], ensure_ascii=False, indent=2) + "\n")
            try:
                if count:
                    self.export_sdf()
                    summary += " SDF 已导出，可下载 ZIP。"
                else:
                    summary += " 没有可导出的成功分子。"
            except Exception as exc:
                summary += f" 导出失败：{exc}；请在清理前重试导出。"
                self.error()
            finally:
                self.busy = False
                self.refresh()
                self.message(summary)

    def export_sdf(self):
        successes = self.manifest.get("successes", [])
        if not successes:
            raise ValueError("当前没有成功生成的分子")
        sources = [Path(item["reconstruction_outputs"]["relaxed_ligand_sdf"]) for item in successes]
        missing = [str(path) for path in sources if not path.is_file()]
        if missing:
            raise FileNotFoundError("成功结果缺少 SDF：" + ", ".join(missing))
        if self.export_dir is None:
            self.export_dir = (EXPORT_ROOT / uuid.uuid4().hex).resolve()
        self.export_dir.mkdir(parents=True, exist_ok=True)
        files = []
        for index, source in enumerate(sources, 1):
            destination = self.export_dir / f"molecule_{index:04d}.sdf"
            shutil.copy2(source, destination)
            files.append(destination)
        archive = self.export_dir / "generated_molecules.zip"
        with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
            for path in files:
                bundle.write(path, arcname=path.name)
        # 内嵌下载链接不依赖 Jupyter 的根目录或服务器文件路由。
        encoded = base64.b64encode(archive.read_bytes()).decode("ascii")
        table = "".join(f"<tr><td>{path.name}</td><td>{item.get('seed', '')}</td></tr>"
                        for path, item in zip(files, successes))
        content = (f'<a download="generated_molecules.zip" href="data:application/zip;base64,{encoded}">'
                   f'下载全部 {len(files)} 个分子的 SDF（ZIP）</a>'
                   f'<p>持久保存目录：{html.escape(str(self.export_dir))}</p>'
                   f'<table><tr><th>SDF 文件</th><th>随机种子</th></tr>{table}</table>')
        self.results.clear_output(wait=True)
        self.results.append_display_data(HTML(content))
        self.message(f"已导出全部 {len(files)} 个成功分子，清理缓存不会删除这些导出文件。")

    def clear_cache(self):
        # 只删本面板拥有的两个子目录；输入、导出、权重和公共词表均不在范围内。
        for target, root in ((self.condition_dir, CONDITION_ROOT), (self.inference_dir, INFERENCE_ROOT)):
            resolved = target.resolve()
            if resolved.parent != Path(root).resolve() or not resolved.name.startswith("notebook_"):
                raise ValueError(f"缓存目录不在预期位置：{resolved}")
            if resolved.exists():
                shutil.rmtree(resolved)
        self.state = self.description = None
        self.invalidate_condition()
        self.clear_fragments()
        self.manifest = {}
        self.run_dir = self.cancel_path = None
        self.viewer.clear_output()
        self.input_info.value = self.location_info.value = ""
        self.steps.selected_index = 0
        self.logs.clear_output()
        self.details.value = ""
        self.progress.value = 0
        self.progress.bar_style = ""
        # 保留最近导出链接，磁盘上所有成功导出文件也一直保留。
        self.message("本面板的预处理和推理缓存已清除；输入文件和已导出 SDF/ZIP 保留。可重新读取输入。")

    def go_to(self, index):
        if not self.busy:
            self.steps.selected_index = index

    def step_changed(self, change):
        if self.busy or self.updating or self.state is None:
            return
        if change["new"] == 1:
            self.action(self.preview)
        elif change["new"] == 2 and "reference_center" in self.state:
            self.action(self.render_fragments)
        elif change["new"] == 3 and self.condition is not None:
            self.action(self.show_condition)

    def kept_ids(self):
        return {box.shard_id for box in self.checkboxes if box.value}

    def clear_fragments(self):
        self.shards, self.checkboxes = [], []
        self.shard_box.children = ()
        self.shard_viewer.clear_output()
        self.selection_info.value = ""

    def reference_geometry(self):
        """预览直接复用当前预处理模块的受体清理与距离选择函数。"""
        parsed = self.state["parsed"]
        choice = self.location.value
        if choice == "sdf":
            coords = self.state["sdf_mol"].GetConformer(0).GetPositions()
            receptor_keys, _, _ = separator.prepare_sdf_receptor_groups(parsed)
            radius, label = separator.TASKA_POCKET_RADIUS, "SDF 配体"
        elif choice == "coordinates":
            coords = np.array([[control.value for control in self.xyz]], dtype=float)
            if not np.isfinite(coords).all():
                raise ValueError("口袋坐标必须是有限数值")
            receptor_keys, _, _ = separator.prepare_sdf_receptor_groups(parsed)
            radius, label = separator.TASKB_POCKET_RADIUS, "手动坐标"
        else:
            candidate = self.description["ligand_candidates"][int(choice)]
            key = tuple(candidate["key"])
            coords = separator.group_coordinates(parsed["groups"][key])
            receptor_keys, _, _ = separator.prepare_receptor_groups(parsed, selected_ligand_key=key)
            radius, label = separator.TASKA_POCKET_RADIUS, candidate["label"]
        center = self.state.get("reference_center")
        if center is None:
            center, _ = separator.utils.calculate_minimum_enclosing_ball(coords)
        pocket_keys = separator.select_groups_within_radius(parsed, receptor_keys, coords, radius)
        return np.asarray(center, dtype=float), radius, label, pocket_keys

    def render_fragments(self):
        if self.task.value == "TASKB":
            self.shard_viewer.clear_output()
            self.selection_info.value = "<b>TASKB：参考配体仅用于定位，固定配体节点为 0，不进行片段保留。</b>"
            return
        kept = self.kept_ids()
        self.selection_info.value = f"已保留 {len(kept)} / {len(self.shards)} 个片段。彩色为保留，灰色为不保留。"
        if not kept:
            self.selection_info.value += "<br><b>TASKA 至少保留一个片段；若仅用配体定位，请点击“仅定位，转为 TASKB”。</b>"
        view = new_view()
        add_fragments(view, self.shards, kept)
        view.zoomTo()
        show_view(self.shard_viewer, view)

    def select_fragments(self, selected):
        self.updating = True
        try:
            for box in self.checkboxes:
                box.value = selected
        finally:
            self.updating = False
        self.shard_changed(None)

    def switch_to_taskb(self):
        # 保持当前文件和定位选择，明确转为无固定配体节点的 TASKB。
        self.updating = True
        try:
            self.task.value = "TASKB"
        finally:
            self.updating = False
        self.prepare()
        if not self.state.get("sdf_path"):
            selected = self.location.value
            self.updating = True
            try:
                self.location.options = list(self.location.options) + [("手动坐标（中心周围 12 Å）", "coordinates")]
                self.location.value = selected
                for control, value in zip(self.xyz, self.state["reference_center"]):
                    control.value = float(value)
            finally:
                self.updating = False

    def show_condition(self):
        condition, view = self.condition, new_view()
        view.addModel(Path(condition["outputs"]["pocket_pdb"]).read_text(encoding="utf-8"), "pdb")
        view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray", "opacity": 0.6},
                                    "stick": {"colorscheme": "Jmol", "radius": 0.1}})
        kept = self.kept_ids()
        if self.task.value == "TASKA":
            add_fragments(view, [shard for shard in self.shards if shard["shard_id"] in kept], kept,
                          offset=condition["metadata"]["reference_center_original"])
        add_label(view, "中心 (0, 0, 0)", [0, 0, 0], "#a85030")
        view.zoomTo()
        show_view(self.condition_viewer, view)


# 用户运行此单元格打开生成面板；重复运行时复用面板，保留当前任务状态。
if "generation_panel" not in globals():
    generation_panel = GenerationPanel()
display(generation_panel.ui)

## 2. 抗结核活性预测

用户完成生成后运行下面两个单元格，点击 **读取生成分子 → 开始预测**。
直接读取本次导出的 SDF，无需下载后重新上传；也可通过文件上传功能测试已有分子的活性。


In [ ]:
import gc
import math
import shutil
import sys
import uuid
import warnings
from html import escape
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "PhiSGATv2").is_dir():
    raise FileNotFoundError("请在项目根目录打开 Notebook，或修改 PROJECT_ROOT。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("当前内核缺少 ipywidgets，请使用配置好交互控件的 Jupyter 环境。") from exc

from IPython.display import FileLink, Image, display

# 仅屏蔽 PyG GlobalAttention 的弃用提示，保留其他警告。
warnings.filterwarnings(
    "ignore",
    message=r"^'nn\.glob\.GlobalAttention' is deprecated, use 'nn\.aggr\.AttentionalAggregation' instead$",
    category=UserWarning,
    module=r"^torch_geometric\.deprecation$",
)

# 先导入预处理模块，让它按现有逻辑设置数值库线程环境。
from PhiSSeparator import PhiSSeparator_2D_work_data_process as process_2d
from PhiSGATv2 import PhiSGATv2_config as config
from PhiSGATv2 import PhiSGATv2_predict as predictor
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D

CHECKPOINT = Path(config.BEST_CHECKPOINT_PATH)
DEVICE = None  # None 沿用项目配置；也可设置为 "cuda:0" 或 "cpu"。
PAGE_SIZE = 20
GRID_COLUMNS = 4
MOLECULE_SIZE = (300, 240)

print(f"项目目录：{PROJECT_ROOT}")
print(f"预测权重：{CHECKPOINT}")
print(f"推理设备：{DEVICE or config.DEVICE}")

In [ ]:
class ActivityNotebook:
    def __init__(self):
        self.session_id = f"notebook_{uuid.uuid4().hex}"
        self.roots = {
            "input": Path(config.WORK_INPUT_DIR).resolve(),
            "pyg": Path(config.WORK_PYG_DIR).resolve(),
            "results": Path(config.PREDICTION_OUTPUT_DIR).resolve(),
        }
        self.paths = {key: root / self.session_id for key, root in self.roots.items()}
        self.busy = False
        self.records = []
        self.image_paths = []
        self.generated_files = []
        self.use_generated = widgets.Button(description="读取生成分子", button_style="info")
        self.upload = widgets.FileUpload(
            accept=".sdf,.csv,.xlsx", multiple=True, description="选择本地文件"
        )
        self.run_button = widgets.Button(description="开始预测", button_style="success", disabled=True)
        self.clear_button = widgets.Button(description="清除缓存", button_style="warning")
        self.page = widgets.Dropdown(options=[], description="结果页：", disabled=True)
        self.status = widgets.HTML(value="请选择文件，然后点击开始预测。")
        self.logs = widgets.Output()
        self.summary = widgets.Output()
        self.grid = widgets.Output()
        locations = "<br>".join(
            f"{label}：<code>{escape(str(self.paths[key]))}</code>"
            for key, label in (("input", "输入"), ("pyg", "预处理"), ("results", "预测结果"))
        )
        self.view = widgets.VBox([
            widgets.HTML(value="<h3>分子活性预测</h3>支持 SDF、CSV、XLSX；可一次选择多个文件。"),
            widgets.HBox([self.use_generated, self.upload, self.run_button, self.clear_button]),
            widgets.HTML(value="清除缓存将删除本会话所有批次的 PyG、结果 CSV、图片及处理报告；输入文件保留。"),
            self.status, widgets.HTML(value=locations), self.summary,
            self.page, self.grid,
            widgets.HTML(value="<b>处理日志</b>"), self.logs,
        ])
        self.use_generated.on_click(self.logs.capture()(self._use_generated))
        self.upload.observe(self._on_upload, names="value")
        self.page.observe(self._show_page, names="value")
        self.run_button.on_click(self.logs.capture()(self._run))
        self.clear_button.on_click(self._clear)

    def _uploads(self):
        if self.generated_files:
            return [(path.name, path.read_bytes()) for path in self.generated_files]
        value = self.upload.value
        # FileUpload 的两种公开数据格式：文件名字典或文件记录序列。
        if isinstance(value, dict):
            return [(item.get("metadata", {}).get("name", name), item["content"])
                    for name, item in value.items()]
        return [(item["name"], item["content"]) for item in value]

    # 读取用户当前生成批次的 SDF；活性预测仍需用户点击按钮启动。
    def _use_generated(self, button):
        if self.busy:
            return
        self.generated_files = []
        self.upload.value = {} if isinstance(self.upload.value, dict) else ()
        self.run_button.disabled = True
        panel = globals().get("generation_panel")
        if panel is None or panel.busy or panel.export_dir is None:
            self.status.value = "请先完成分子生成并导出 SDF。"
            return
        files = sorted(panel.export_dir.glob("*.sdf"))
        if not files:
            self.status.value = "没有成功导出的 SDF，请先生成分子。"
            return
        self.generated_files = files
        self.run_button.disabled = False
        self.status.value = escape(f"已读取 {len(files)} 个生成分子：{panel.export_dir}；点击开始预测。")

    def _on_upload(self, change):
        self.generated_files = []
        self.run_button.disabled = self.busy or not bool(self.upload.value)
        if not self.busy:
            names = [str(name) for name, _ in self._uploads()]
            self.status.value = escape("已选择：" + "、".join(names) if names else "请选择文件。")

    def _set_busy(self, busy):
        self.busy = busy
        self.upload.disabled = busy
        self.use_generated.disabled = busy
        self.run_button.disabled = busy or not (self.generated_files or self.upload.value)
        self.clear_button.disabled = busy
        self.page.disabled = busy or not bool(self.page.options)

    def _reset_display(self):
        self.records = []
        self.image_paths = []
        self.page.options = []
        self.page.disabled = True
        self.summary.clear_output()
        self.grid.clear_output()

    def _save_grids(self, image_dir):
        image_dir.mkdir(parents=True, exist_ok=True)
        for start in range(0, len(self.records), PAGE_SIZE):
            molecules, legends = [], []
            path = None
            for rank, record in enumerate(self.records[start:start + PAGE_SIZE], start=start + 1):
                try:
                    mol = Chem.MolFromSmiles(record["smiles"])
                    if mol is None:
                        raise ValueError("无法解析 SMILES")
                    for atom in mol.GetAtoms():
                        atom.SetAtomMapNum(0)
                    mol = rdMolDraw2D.PrepareMolForDrawing(mol, addChiralHs=False)
                except Exception as exc:
                    print(f"绘图跳过 {record['compound_id']}：{exc}")
                    continue
                molecules.append(mol)
                # 网格使用排序编号，与下方表格对应，避免长名称或中文字体影响绘图。
                legends.append(f"#{rank} | Activity: {record['predicted_activity']:.4f}")
            if molecules:
                try:
                    columns = min(GRID_COLUMNS, len(molecules))
                    rows = math.ceil(len(molecules) / columns)
                    width, height = MOLECULE_SIZE
                    drawer = rdMolDraw2D.MolDraw2DCairo(columns * width, rows * height, width, height)
                    drawer.drawOptions().prepareMolsBeforeDrawing = False
                    drawer.DrawMolecules(molecules, legends=legends)
                    drawer.FinishDrawing()
                    candidate = image_dir / f"molecules_{start // PAGE_SIZE + 1:03d}.png"
                    candidate.write_bytes(drawer.GetDrawingText())
                    path = candidate
                except Exception as exc:
                    print(f"本页网格绘制失败，预测数值仍保留：{exc}")
            self.image_paths.append(path)

    def _show_page(self, change=None):
        self.grid.clear_output(wait=True)
        index = self.page.value
        if index is None or not self.records:
            return
        start = index * PAGE_SIZE
        records = self.records[start:start + PAGE_SIZE]
        with self.grid:
            path = self.image_paths[index]
            if path is not None and path.is_file():
                display(Image(filename=str(path), width=1000))
            table = pd.DataFrame(records)[["compound_id", "smiles", "predicted_activity"]]
            table.index = range(start + 1, start + len(table) + 1)
            table.index.name = "网格编号"
            display(table)

    # 用户点击“开始预测”后，执行二维预处理和活性预测。
    def _run(self, button):
        if self.busy or not (self.generated_files or self.upload.value):
            return
        self._set_busy(True)
        self._reset_display()
        self.logs.clear_output()
        try:
            if not CHECKPOINT.is_file():
                raise FileNotFoundError(f"预测权重不存在：{CHECKPOINT}")
            batch_id = f"batch_{uuid.uuid4().hex[:12]}"
            paths = {key: root / batch_id for key, root in self.paths.items()}
            for path in paths.values():
                path.mkdir(parents=True, exist_ok=False)
            for index, (name, content) in enumerate(self._uploads(), start=1):
                filename = Path(str(name).replace("\\", "/")).name
                if Path(filename).suffix.lower() not in {".sdf", ".csv", ".xlsx"}:
                    raise ValueError(f"不支持的文件类型：{filename}")
                # 同名文件分别放入子目录，避免上传时互相覆盖。
                source_dir = paths["input"] / f"file_{index:03d}"
                source_dir.mkdir()
                (source_dir / filename).write_bytes(bytes(content))

            self.status.value = "正在预处理待测分子……"
            summary = process_2d.process_work_data(
                input_dir=paths["input"], output_dir=paths["pyg"], num_workers=1,
            )
            if not summary["saved_pyg"]:
                raise RuntimeError("没有成功生成可预测的 PyG，请查看处理错误报告。")
            self.status.value = "正在预测分子活性……"
            result_path = paths["results"] / "prediction_results.csv"
            error_path = paths["results"] / "prediction_errors.csv"
            records = predictor.predict_activity(
                data_dir=paths["pyg"], checkpoint_path=CHECKPOINT, device=DEVICE,
                result_path=result_path, error_path=error_path,
                image_dir=None, data_loader_workers=0,
            )
            self.records = sorted(records, key=lambda row: -float(row["predicted_activity"]))
            self.status.value = "正在生成分子网格……"
            self._save_grids(paths["results"] / "molecule_images")
            self.page.options = [(f"第 {i + 1} 页", i) for i in range(len(self.image_paths))]
            self.page.value = 0
            self._show_page()
            error_count = len(pd.read_csv(error_path))
            with self.summary:
                print(f"本批次：{batch_id}")
                print(f"预处理成功 {summary['saved_pyg']} 个；预处理错误 {summary['error_count']} 条；"
                      f"预测成功 {len(records)} 个；预测错误 {error_count} 条。")
                print("网格按预测值降序展示；CSV 保留底层原始顺序及完整数值。")
                print(f"输入：{paths['input']}")
                print(f"预处理：{paths['pyg']}")
                print(f"结果：{paths['results']}")
                for path in (result_path, error_path, paths["pyg"] / "processing_errors.csv"):
                    if path.is_file():
                        # 默认目录位于 Notebook 根目录下，可通过 Jupyter 文件服务打开。
                        if path.is_relative_to(PROJECT_ROOT):
                            display(FileLink(str(path.relative_to(PROJECT_ROOT))))
                        else:
                            print(path)
            self.status.value = f"完成：已预测 {len(records)} 个分子，可选择结果页查看。"
        except (Exception, KeyboardInterrupt) as exc:
            with self.logs:
                print(f"任务未完成：{type(exc).__name__}: {exc}")
                if "paths" in locals():
                    print(f"本批次预处理和结果目录：{paths['pyg']}；{paths['results']}")
            self.status.value = "任务未完成，请查看日志；本次文件可通过清除缓存按钮清理。"
        finally:
            self._set_busy(False)

    def _clear(self, button):
        if self.busy:
            return
        self._set_busy(True)
        failures = []
        try:
            # 仅删除本对象创建的两个会话目录。先核对所有目标，再执行删除。
            targets = [(self.roots[key], self.paths[key]) for key in ("pyg", "results")]
            for root, path in targets:
                if path.is_symlink() or path.resolve().parent != root or path.name != self.session_id:
                    raise ValueError(f"清理目标不是本会话目录：{path}")
            for root, path in targets:
                try:
                    if path.exists():
                        shutil.rmtree(path)
                except OSError as exc:
                    failures.append(f"{path}：{exc}")
            self._reset_display()
            self.logs.clear_output()
            self.generated_files = []
            self.upload.value = {} if isinstance(self.upload.value, dict) else ()
            gc.collect()
            if failures:
                with self.logs:
                    print("部分文件未能清除，可再次点击清除缓存：\n" + "\n".join(failures))
                self.status.value = "部分缓存未清除，请查看日志。"
            else:
                self.status.value = "本会话预处理与预测缓存已清除，输入文件已保留。可重新选择文件。"
        except Exception as exc:
            self.status.value = escape(f"清理停止：{exc}")
        finally:
            self._set_busy(False)

# 用户运行此单元格打开预测面板；重复运行时复用当前会话。
if "phistone_activity_ui" not in globals():
    phistone_activity_ui = ActivityNotebook()
display(phistone_activity_ui.view)

SDF 和 ZIP 保存在 `Datas/work_datas/generated_sdf/`；预测 CSV 和结构网格保存在 `Datas/work_datas/PhiSGATv2_prediction_results/`。各面板显示本次实际路径。

预测值保留模型原始输出，不代表实验测量。清理预测缓存前先保存 CSV。

## 3. 保存本次测试结果

用户完成预测后、清理缓存前，运行下面单元格，再保存 Notebook（保留输出）并下载。生成数量、预测表格和分子图片将作为普通输出保存在文件中，供本地查看。


In [ ]:
# 用户完成预测后运行本单元格，保存生成数量、活性表格和分子图片。
print("成功生成分子数：", len(generation_panel.manifest.get("successes", [])))
print("活性预测分子数：", len(phistone_activity_ui.records))
display(pd.DataFrame(phistone_activity_ui.records))

for path in phistone_activity_ui.image_paths:
    if path is not None and path.is_file():
        display(Image(filename=str(path), embed=True))
